# Final Confirmatory Evaluation Runner

This notebook consumes one completed final confirmatory benchmark run and emits final aggregated reporting artifacts.

It expects a manifest with run kind final_confirmatory_benchmark and evaluates fixed-loss outputs without robustness expansion by default.

In [ ]:
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    repo_markers = ("pyproject.toml", ".git")
    package_marker = Path("ml_model") / "__init__.py"
    training_marker = Path("ml_model") / "notebooks" / "training" / "io.py"
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in repo_markers) and (candidate / training_marker).exists():
            return candidate
    absolute_fallback = Path(r"C:/Users/Froi/Documents/project/injection-alert-system")
    if (absolute_fallback / package_marker).exists() and (absolute_fallback / training_marker).exists():
        return absolute_fallback
    raise FileNotFoundError(f"Could not locate repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ml_model.preprocessing.dataset_io import (
    evaluation_dir,
    latest_run_dir,
    load_json,
    load_numpy_artifacts,
    save_csv,
    save_json,
)
from ml_model.evaluation.metrics import aggregate_numeric_columns, aggregate_per_class_metrics, evaluate_from_logits

In [ ]:
DATASET_VERSION = "v3_907k_cleaned"
ECE_N_BINS = 15
CONFIDENCE_THRESHOLDS = [0.5, 0.7, 0.8, 0.9]
EVAL_ENABLE_ROBUSTNESS = os.getenv("BENCHMARK_EVAL_ENABLE_ROBUSTNESS", "0").strip().lower() in {
    "1",
    "true",
    "yes",
    "on",
}
EXPECTED_RUN_KIND = "final_confirmatory_benchmark"
FINAL_RESULTS_BASE_DIR = REPO_ROOT / "ml_model" / "notebooks" / "training done" / "Final training" / "results"

run_dir_override = os.getenv("BENCHMARK_RUN_DIR", "").strip()
if run_dir_override:
    RUN_DIR = Path(run_dir_override).expanduser().resolve()
else:
    RUN_DIR = latest_run_dir(base_dir=FINAL_RESULTS_BASE_DIR, dataset_version=DATASET_VERSION)

if not RUN_DIR.exists():
    raise FileNotFoundError(f"Run directory does not exist: {RUN_DIR}")

EVALUATION_OUTPUT_DIR = evaluation_dir(RUN_DIR)
MANIFEST_PATH = RUN_DIR / "run_manifest.json"
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing run manifest: {MANIFEST_PATH}")

manifest = load_json(MANIFEST_PATH)
source_run_kind = manifest.get("run_kind")
if source_run_kind != EXPECTED_RUN_KIND:
    raise ValueError(
        f"Expected run_kind={EXPECTED_RUN_KIND}, found {source_run_kind}. "
        f"Run dir is not a final confirmatory benchmark: {RUN_DIR}"
    )

DATASET_VERSION = manifest.get("dataset_version", DATASET_VERSION)
LABEL_NAMES = [str(x) for x in manifest.get("label_names", [])]
FIXED_LOSS_KEY = str(manifest.get("fixed_loss_key", "")).strip()

manifest_model_keys = manifest.get("model_keys")
if isinstance(manifest_model_keys, (list, tuple, set)):
    model_keys = [str(key) for key in manifest_model_keys]
else:
    model_keys = sorted(p.name for p in RUN_DIR.iterdir() if p.is_dir() and p.name != "evaluation")

if not model_keys:
    raise FileNotFoundError(f"No model artifact directories found in run directory: {RUN_DIR}")

print(f"Run dir            : {RUN_DIR}")
print(f"Evaluation dir     : {EVALUATION_OUTPUT_DIR}")
print(f"Manifest path      : {MANIFEST_PATH}")
print(f"Run kind           : {source_run_kind}")
print(f"Model keys         : {model_keys}")
print(f"Fixed loss key     : {FIXED_LOSS_KEY}")
print(f"Robustness enabled : {EVAL_ENABLE_ROBUSTNESS} (disabled in first-pass final evaluation)")

Run dir            : C:\Users\Froi\Documents\project\injection-alert-system\ml_model\notebooks\training\results\v3_907k_cleaned_screening_weighted_ce_calibrated_security_20260409_205942
Evaluation dir     : C:\Users\Froi\Documents\project\injection-alert-system\ml_model\notebooks\training\results\v3_907k_cleaned_screening_weighted_ce_calibrated_security_20260409_205942\evaluation
Data dir           : C:\Users\Froi\Documents\project\injection-alert-system\data\processed\v3_907k_cleaned
Model keys         : ['distilbert', 'minilm_l6', 'tinybert_bigru_attn']
Manifest loaded    : True
Label names loaded : ['Code Injection', 'Normal', 'Other Attacks', 'SQL Injection']
Evaluation smoke   : False
Robustness enabled : True


In [ ]:
def coerce_float(value, default=np.nan):
    if value is None:
        return float(default)
    try:
        return float(value)
    except (TypeError, ValueError):
        return float(default)


def parse_seed(seed_dir: Path) -> int:
    try:
        return int(seed_dir.name.split("_")[-1])
    except Exception:
        return -1


def resolve_loss_key_for_model(model_key: str) -> str:
    model_dir = RUN_DIR / model_key
    if FIXED_LOSS_KEY:
        expected_dir = model_dir / f"loss_{FIXED_LOSS_KEY}"
        if expected_dir.exists():
            return FIXED_LOSS_KEY
        raise FileNotFoundError(
            f"Manifest fixed_loss_key '{FIXED_LOSS_KEY}' not found for model {model_key} at {expected_dir}"
        )

    loss_dirs = sorted([p for p in model_dir.glob("loss_*") if p.is_dir()])
    if len(loss_dirs) == 1:
        return loss_dirs[0].name.replace("loss_", "", 1)

    raise FileNotFoundError(
        f"Could not resolve fixed loss for model {model_key}. "
        f"Manifest fixed_loss_key missing and found {len(loss_dirs)} loss directories."
    )


def seed_dirs_for_model(model_key: str, loss_key: str) -> list[Path]:
    variant_dir = RUN_DIR / model_key / f"loss_{loss_key}"
    if not variant_dir.exists():
        raise FileNotFoundError(f"Missing variant directory: {variant_dir}")

    seed_dirs = sorted([p for p in variant_dir.glob("seed_*") if p.is_dir()])
    if not seed_dirs:
        raise FileNotFoundError(f"No seed directories found in {variant_dir}")
    return seed_dirs


def load_seed_artifacts(seed_dir: Path):
    summary = load_json(seed_dir / "summary_metrics.json") if (seed_dir / "summary_metrics.json").exists() else {}
    config = load_json(seed_dir / "config_metadata.json") if (seed_dir / "config_metadata.json").exists() else {}

    temperature = 1.0
    calibration_path = seed_dir / "calibration.json"
    if calibration_path.exists():
        calibration = load_json(calibration_path)
        temperature = float(calibration.get("temperature", 1.0))

    val_outputs = load_numpy_artifacts(seed_dir / "validation_outputs.npz")
    test_outputs = load_numpy_artifacts(seed_dir / "test_outputs.npz")

    return {
        "summary": summary,
        "config": config,
        "temperature": temperature,
        "val_logits": val_outputs["logits"],
        "val_labels": val_outputs["labels"].astype(np.int64),
        "test_logits": test_outputs["logits"],
        "test_labels": test_outputs["labels"].astype(np.int64),
    }


comparison_rows = []
aggregated_per_class_tables = []
latency_rows = []
consumed_artifacts = set()

for model_key in model_keys:
    try:
        loss_key = resolve_loss_key_for_model(model_key)
        seed_dirs = seed_dirs_for_model(model_key, loss_key)
    except FileNotFoundError as exc:
        print(f"Skipping {model_key}: {exc}")
        continue

    model_seed_rows = []
    per_class_seed_frames = []
    model_latency_seed_rows = []

    for seed_dir in seed_dirs:
        seed = parse_seed(seed_dir)
        artifacts = load_seed_artifacts(seed_dir)

        val_uncal = evaluate_from_logits(artifacts["val_logits"], artifacts["val_labels"], n_bins=ECE_N_BINS)
        test_uncal = evaluate_from_logits(artifacts["test_logits"], artifacts["test_labels"], n_bins=ECE_N_BINS)
        val_cal = evaluate_from_logits(
            artifacts["val_logits"] / artifacts["temperature"],
            artifacts["val_labels"],
            n_bins=ECE_N_BINS,
        )
        test_cal = evaluate_from_logits(
            artifacts["test_logits"] / artifacts["temperature"],
            artifacts["test_labels"],
            n_bins=ECE_N_BINS,
        )

        model_seed_rows.append(
            {
                "model_key": model_key,
                "loss_key": loss_key,
                "seed": int(seed),
                "architecture": artifacts["summary"].get("architecture", artifacts["config"].get("architecture", "unknown")),
                "architecture_family": artifacts["summary"].get("architecture_family", "unknown"),
                "head_type": artifacts["summary"].get("head_type", "unknown"),
                "experiment_phase": artifacts["summary"].get("experiment_phase", "unknown"),
                "temperature": coerce_float(artifacts["temperature"]),
                "val_accuracy": coerce_float(val_uncal["accuracy"]),
                "val_balanced_accuracy": coerce_float(val_uncal["balanced_accuracy"]),
                "val_macro_f1": coerce_float(val_uncal["macro_f1"]),
                "val_weighted_f1": coerce_float(val_uncal["weighted_f1"]),
                "val_ece_uncalibrated": coerce_float(val_uncal["ece"]),
                "val_ece_calibrated": coerce_float(val_cal["ece"]),
                "val_nll_uncalibrated": coerce_float(val_uncal["nll"]),
                "val_nll_calibrated": coerce_float(val_cal["nll"]),
                "val_brier_uncalibrated": coerce_float(val_uncal["brier_score"]),
                "val_brier_calibrated": coerce_float(val_cal["brier_score"]),
                "test_accuracy": coerce_float(test_uncal["accuracy"]),
                "test_balanced_accuracy": coerce_float(test_uncal["balanced_accuracy"]),
                "test_macro_f1": coerce_float(test_uncal["macro_f1"]),
                "test_weighted_f1": coerce_float(test_uncal["weighted_f1"]),
                "test_ece_uncalibrated": coerce_float(test_uncal["ece"]),
                "test_ece_calibrated": coerce_float(test_cal["ece"]),
                "test_nll_uncalibrated": coerce_float(test_uncal["nll"]),
                "test_nll_calibrated": coerce_float(test_cal["nll"]),
                "test_brier_uncalibrated": coerce_float(test_uncal["brier_score"]),
                "test_brier_calibrated": coerce_float(test_cal["brier_score"]),
                "normal_false_positive_rate": coerce_float(artifacts["summary"].get("normal_false_positive_rate")),
                "attack_escape_rate": coerce_float(artifacts["summary"].get("attack_escape_rate")),
                "inference_latency_mean_ms": coerce_float(artifacts["summary"].get("inference_latency_mean_ms")),
                "inference_latency_std_ms": coerce_float(artifacts["summary"].get("inference_latency_std_ms")),
                "inference_latency_p50_ms": coerce_float(artifacts["summary"].get("inference_latency_p50_ms")),
                "inference_latency_p95_ms": coerce_float(artifacts["summary"].get("inference_latency_p95_ms")),
                "model_size_mb": coerce_float(artifacts["summary"].get("model_size_mb")),
                "training_workflow_runtime_sec": coerce_float(artifacts["summary"].get("training_workflow_runtime_sec")),
                "mean_epoch_training_time_sec": coerce_float(artifacts["summary"].get("mean_epoch_training_time_sec")),
            }
        )

        per_class_path = seed_dir / "per_class_metrics.json"
        if per_class_path.exists():
            per_class_df = pd.DataFrame(load_json(per_class_path))
            if not per_class_df.empty:
                per_class_df["seed"] = int(seed)
                per_class_seed_frames.append(per_class_df)

        latency_path = seed_dir / "latency_summary.json"
        if latency_path.exists():
            latency_payload = load_json(latency_path)
            model_latency_seed_rows.append(
                {
                    "seed": int(seed),
                    "inference_latency_mean_ms": coerce_float(latency_payload.get("latency_mean_ms")),
                    "inference_latency_std_ms": coerce_float(latency_payload.get("latency_std_ms")),
                    "inference_latency_p50_ms": coerce_float(latency_payload.get("latency_p50_ms")),
                    "inference_latency_p95_ms": coerce_float(latency_payload.get("latency_p95_ms")),
                    "inference_latency_min_ms": coerce_float(latency_payload.get("latency_min_ms")),
                    "inference_latency_max_ms": coerce_float(latency_payload.get("latency_max_ms")),
                    "latency_n_measurements": coerce_float(latency_payload.get("n_measurements")),
                    "latency_batch_size": coerce_float(latency_payload.get("batch_size")),
                    "latency_sequence_length": coerce_float(latency_payload.get("sequence_length")),
                }
            )

        consumed_artifacts.update(
            {
                str(seed_dir / "summary_metrics.json"),
                str(seed_dir / "config_metadata.json"),
                str(seed_dir / "calibration.json"),
                str(seed_dir / "validation_outputs.npz"),
                str(seed_dir / "test_outputs.npz"),
                str(seed_dir / "per_class_metrics.json"),
                str(seed_dir / "latency_summary.json"),
            }
        )

    if not model_seed_rows:
        continue

    seed_df = pd.DataFrame(model_seed_rows).sort_values(by="seed").reset_index(drop=True)
    save_csv(seed_df, EVALUATION_OUTPUT_DIR / f"{model_key}_{loss_key}_seed_metrics.csv", index=False)

    model_aggregate = {
        "model_key": model_key,
        "loss_key": loss_key,
        "n_seeds": int(seed_df.shape[0]),
        "architecture": str(seed_df.iloc[0]["architecture"]),
        "architecture_family": str(seed_df.iloc[0]["architecture_family"]),
        "head_type": str(seed_df.iloc[0]["head_type"]),
        "experiment_phase": str(seed_df.iloc[0]["experiment_phase"]),
        **aggregate_numeric_columns(seed_df, exclude=["seed"]),
    }
    comparison_rows.append(model_aggregate)

    per_class_agg_df = aggregate_per_class_metrics(per_class_seed_frames)
    if not per_class_agg_df.empty:
        per_class_agg_df.insert(0, "loss_key", loss_key)
        per_class_agg_df.insert(0, "model_key", model_key)
        aggregated_per_class_tables.append(per_class_agg_df)

    if model_latency_seed_rows:
        latency_seed_df = pd.DataFrame(model_latency_seed_rows).sort_values(by="seed").reset_index(drop=True)
        save_csv(latency_seed_df, EVALUATION_OUTPUT_DIR / f"{model_key}_{loss_key}_latency_seed_metrics.csv", index=False)
        latency_rows.append(
            {
                "model_key": model_key,
                "loss_key": loss_key,
                "n_seeds": int(latency_seed_df.shape[0]),
                **aggregate_numeric_columns(latency_seed_df, exclude=["seed"]),
            }
        )

if not comparison_rows:
    raise RuntimeError("No completed final confirmatory model artifacts were available for evaluation.")

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    by=["test_macro_f1_mean", "val_macro_f1_mean"],
    ascending=[False, False],
).reset_index(drop=True)
save_csv(comparison_df, EVALUATION_OUTPUT_DIR / "final_model_comparison.csv", index=False)

if aggregated_per_class_tables:
    aggregated_per_class_df = pd.concat(aggregated_per_class_tables, ignore_index=True)
    aggregated_per_class_df = aggregated_per_class_df.sort_values(
        by=["model_key", "label_id"],
        ascending=[True, True],
    ).reset_index(drop=True)
else:
    aggregated_per_class_df = pd.DataFrame(
        columns=[
            "model_key",
            "loss_key",
            "label_id",
            "label_name",
            "precision_mean",
            "precision_std",
            "precision_ci95_lower",
            "precision_ci95_upper",
            "recall_mean",
            "recall_std",
            "recall_ci95_lower",
            "recall_ci95_upper",
            "f1_mean",
            "f1_std",
            "f1_ci95_lower",
            "f1_ci95_upper",
            "support_mean",
            "support_std",
        ]
    )
save_csv(aggregated_per_class_df, EVALUATION_OUTPUT_DIR / "aggregated_per_class_summary.csv", index=False)

if latency_rows:
    latency_comparison_df = pd.DataFrame(latency_rows).sort_values(
        by=["inference_latency_mean_ms_mean", "model_key"],
        ascending=[True, True],
    ).reset_index(drop=True)
else:
    latency_comparison_df = pd.DataFrame(
        columns=[
            "model_key",
            "loss_key",
            "n_seeds",
            "inference_latency_mean_ms_mean",
            "inference_latency_mean_ms_std",
            "inference_latency_mean_ms_ci95_lower",
            "inference_latency_mean_ms_ci95_upper",
        ]
    )
save_csv(latency_comparison_df, EVALUATION_OUTPUT_DIR / "latency_comparison.csv", index=False)

summary_payload = {
    "evaluation_kind": "final_confirmatory_evaluation",
    "source_run_kind": source_run_kind,
    "source_run_dir": str(RUN_DIR),
    "evaluation_output_dir": str(EVALUATION_OUTPUT_DIR),
    "dataset_version": DATASET_VERSION,
    "metric_schema_version": manifest.get("artifact_schema_version", "final_confirmatory_benchmark.v1"),
    "model_count": int(comparison_df.shape[0]),
    "generated_outputs": [
        "final_model_comparison.csv",
        "aggregated_per_class_summary.csv",
        "latency_comparison.csv",
        "evaluation_summary.json",
    ],
    "consumed_artifacts": sorted(consumed_artifacts),
    "created_at": datetime.now(timezone.utc).isoformat(),
}
save_json(EVALUATION_OUTPUT_DIR / "evaluation_summary.json", summary_payload)

preview_cols = [
    "model_key",
    "loss_key",
    "test_macro_f1_mean",
    "test_macro_f1_std",
    "test_macro_f1_ci95_lower",
    "test_macro_f1_ci95_upper",
    "test_balanced_accuracy_mean",
    "test_balanced_accuracy_ci95_lower",
    "test_balanced_accuracy_ci95_upper",
    "inference_latency_mean_ms_mean",
]
display(comparison_df[[col for col in preview_cols if col in comparison_df.columns]])

print("Saved:")
print(f"- {EVALUATION_OUTPUT_DIR / 'final_model_comparison.csv'}")
print(f"- {EVALUATION_OUTPUT_DIR / 'aggregated_per_class_summary.csv'}")
print(f"- {EVALUATION_OUTPUT_DIR / 'latency_comparison.csv'}")
print(f"- {EVALUATION_OUTPUT_DIR / 'evaluation_summary.json'}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nreimers/MiniLM-L6-H384-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_6L_768D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.bias      | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,model_key,experiment_phase,loss_key,architecture,head_type,n_seeds,val_macro_f1_mean,val_macro_f1_std,test_macro_f1_mean,test_macro_f1_std,test_ece_uncalibrated_mean,test_ece_calibrated_mean,test_nll_uncalibrated_mean,test_nll_calibrated_mean,normal_false_positive_rate_mean,attack_escape_rate_mean,inference_latency_ms_mean,model_size_mb_mean
0,tinybert_bigru_attn,architecture_search,weighted_ce,tinybert_bigru_attention,bigru_attention_mlp,1,0.9938,0.0000,0.9884,0.0000,0.0031,0.0051,0.0194,0.0202,0.0014,0.0023,nan,262.18
1,distilbert,controlled_backbone_benchmark,weighted_ce,transformer,mean_pool_mlp,1,0.9941,0.0000,0.9892,0.0000,0.0030,0.0044,0.0175,0.0181,0.0022,0.0020,nan,253.91
2,minilm_l6,controlled_backbone_benchmark,weighted_ce,transformer,mean_pool_mlp,1,0.9919,0.0000,0.9878,0.0000,0.0038,0.0049,0.0224,0.0234,0.0025,0.0033,nan,87.03


Evaluation tables prepared.
Test metrics are reported as final evidence only; no test-set winner selection is performed.
